## Comparison-0-group-wise-protein-groups

> sample-wise

In [1]:
import pandas as pd
import numpy as np
import pyteomics.auxiliary as aux
import seaborn as sns
import matplotlib.pyplot as plt
import os, re, subprocess
from utility_functions import *
DATE, project_palette

('20251015',
 {'canon': 'orangered', 'trembl': 'yellowgreen', 'openprot': 'cornflowerblue'})

In [2]:
import gzip, pickle
with gzip.open('./custom-rank-cutoffs.gz','rb') as infile:
    rank_filters = pickle.load(infile)

In [3]:
working_folder = "C:/Users/Enrico/OneDrive - UGent/run-ionbot"
PXDs = [
    'PXD002057.v0.11.4',
    'PXD005833.v0.11.4',
    'PXD014258.v0.11.4',
    'PXD002057-closed',
    'PXD005833-closed',
    'PXD014258-closed',
]

SEARCHES = [
    'canon',
    'trembl',
    'openprot',
]
DATASETS = pd.MultiIndex.from_product([PXDs,SEARCHES])
DATASETS

MultiIndex([('PXD002057.v0.11.4',    'canon'),
            ('PXD002057.v0.11.4',   'trembl'),
            ('PXD002057.v0.11.4', 'openprot'),
            ('PXD005833.v0.11.4',    'canon'),
            ('PXD005833.v0.11.4',   'trembl'),
            ('PXD005833.v0.11.4', 'openprot'),
            ('PXD014258.v0.11.4',    'canon'),
            ('PXD014258.v0.11.4',   'trembl'),
            ('PXD014258.v0.11.4', 'openprot'),
            ( 'PXD002057-closed',    'canon'),
            ( 'PXD002057-closed',   'trembl'),
            ( 'PXD002057-closed', 'openprot'),
            ( 'PXD005833-closed',    'canon'),
            ( 'PXD005833-closed',   'trembl'),
            ( 'PXD005833-closed', 'openprot'),
            ( 'PXD014258-closed',    'canon'),
            ( 'PXD014258-closed',   'trembl'),
            ( 'PXD014258-closed', 'openprot')],
           )

In [4]:
folders = {search:{dataset_name:[] for dataset_name in PXDs} for search in SEARCHES}
for dataset_name in PXDs:
    for search in SEARCHES:   
        for fld in os.scandir(os.path.join(working_folder, dataset_name, f"{dataset_name}-{search}")):
            if not fld.name.startswith('.') and os.path.isdir(fld.path): 
                folders[search][dataset_name].append(fld)
# folders

In [5]:
def custom_rank_filtering(fld, filters):
    psms = pd.read_csv(os.path.join(fld.path,'ionbot.first.csv'))

    # subfolders should be named: {sample}-{search}
    sample,db = parse_sample_fld_name(fld.name)
    
    psms['proteins'] = psms.proteins.apply(process_proteins)
    psms['leadprot'] = psms.proteins.apply(lambda x: x[0])
    psms['protein_classes'] = psms.proteins.apply(
        lambda lst: np.unique([classify_leadprot(p) for p in lst])
    )
    psms['isCanonical'] = psms.protein_classes.apply(is_peptide_canonical)
    psms['proteins'] = psms.proteins.apply(lambda x: ';'.join(x))
    
    psms.modifications = psms.modifications.fillna('Unmodified')
    psms.unexpected_modification = psms.unexpected_modification.fillna('')
    psms['isModified'] = psms.apply(classifiy_mods, axis=1)
    
    psms = psms[(psms.database=='T')&(psms.isCanonical!='Contam')].copy()
    psms.spectrum_file = psms.spectrum_file.apply(lambda x: x.split('.')[0])
    psms.spectrum_title = psms.spectrum_file + ':' + psms.spectrum_title.apply(lambda x: x.split(':')[-1])

    filtered_results = []
    for c,df in psms.groupby('isCanonical').__iter__():
        maxrank = int( filters[(sample,c,db)] )
        print((sample,c,db), maxrank)
        if maxrank > 0:
            # df.sort_values('psm_score', ascending=False, inplace=True)
            df['rank'] = df['psm_score'].rank(method="dense", ascending=False)
            filtered_results.append(df[df['rank']<=maxrank])
    if len(filtered_results) > 0:
        filtred_results = pd.concat(filtered_results, ignore_index=True)
        filtred_results.to_csv(
            os.path.join(fld.path,'rank-filtered-results.csv.gz'),
            index=False,
            compression='gzip',
            encoding='utf8'
        )
        print(fld.path,'OK')
    else:
        print(fld.path, 'Error!')

In [6]:
for dataset_name in PXDs:
    for search in SEARCHES:  
        j = len(folders[search][dataset_name])
        for i,sample_fld in enumerate(folders[search][dataset_name]):
            custom_rank_filtering(sample_fld, rank_filters)
print('\nDONE!')

('130327_o2_01_hu_C1_2hr', 'Canonical', 'canon') 3118
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\130327_o2_01_hu_C1_2hr-canon OK
('130327_o2_02_hu_P1_2hr', 'Canonical', 'canon') 6238
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\130327_o2_02_hu_P1_2hr-canon OK
('130327_o2_03_hu_C2_2hr', 'Canonical', 'canon') 2353
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\130327_o2_03_hu_C2_2hr-canon OK
('130327_o2_04_hu_P2_2hr', 'Canonical', 'canon') 5327
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\130327_o2_04_hu_P2_2hr-canon OK
('130327_o2_05_hu_C3_2hr', 'Canonical', 'canon') 2505
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\130327_o2_05_hu_C3_2hr-canon OK
('130327_o2_06_hu_P3_2hr', 'Canonical', 'canon') 5680
C:/Users/Enrico/OneDrive - UGent/run-ionbot\PXD002057.v0.11.4\PXD002057.v0.11.4-canon\

-----